# 10 · Robustness of the screen validation (plan 7i), figures and tables

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 31 Oct 2026 · **Needs:** `results/01_inventory/` to `results/09_transfer_posthoc/` · **Plan:**
sections **7i** (added 31 Oct 2026 at Hamza's request) and **8** (figures and tables).

**What this notebook does, in plain words**

1. **Robustness of the screen validation (plan 7i, post hoc sensitivity analysis).** Notebook 07 found that, over 19
   CT tasks, the metadata AUC and the background-only AUC go together (CT-FM: Spearman rho 0.51). Two worries:
   - the correlation may lean on the 4 flagged tasks, so it is recomputed **without them** (15 tasks);
   - several tasks share patients, so it is recomputed over the **11 groups of tasks that share no patients**
     (each group's mean AUCs);
   - and, descriptively, with each task left out in turn.

   The wording of each result was fixed in plan 7i before anything was computed.
2. **The paper's figures and tables**, drawn only from the saved results of notebooks 01–09. Nothing is refitted,
   downloaded or relabelled.

**How to run:** CPU is enough, a few minutes. In Colab: *Runtime → Run all* and sign in with
**the Google account that holds the project Drive folder** when Drive asks. It also runs on a computer that holds a verified copy of `results/`
(set `ASA_PROJECT_DIR`). Outputs go to `results/10_figures_tables/`. In Colab, the last cell stops the runtime.

In [ ]:
# 1. Setup: folders
import os, json, time, hashlib, datetime, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # a local copy of the project folder whose results/ match their run records (checked below)
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')

RES = PROJECT / 'results'
IN = {'01': RES / '01_inventory', '02': RES / '02_tasks_features', '03': RES / '03_audit', '04': RES / '04_posthoc',
      '05': RES / '05_image_check', '06': RES / '06_image_posthoc', '07': RES / '07_screen_validation',
      '08': RES / '08_transfer', '09': RES / '09_transfer_posthoc'}
OUT = RES / '10_figures_tables'
FIG, TAB = OUT / 'figures', OUT / 'tables'
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project folder:', PROJECT)
print('Outputs to    :', OUT)
print('Run started   :', RUN_STARTED, '(UTC)')

In [ ]:
# 2. Packages
import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)

def show(df, n=40):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__}
print(VERSIONS)

## Settings
The task-permutation test is notebook 07's (seed and number of permutations checked against its run record below).
Colours: three validated categorical slots (colour-blind safe as a set), with the meaning fixed across figures:
blue = full image (or "not flagged"), orange = background only (or "flagged"), aqua = silhouette.

In [ ]:
# 3. Settings
SEED = 2026
N_TASK_PERM = 99_999          # notebook 07's test across tasks (plan 7g, A.4(a))
ENCODERS = ('ctfm', 'merlin')
ENC = {'ctfm': 'CT-FM', 'merlin': 'Merlin'}

INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INPUT_COLOR = {'image': BLUE, 'background': ORANGE, 'silhouette': AQUA}
INPUT_NAME = {'image': 'full image', 'background': 'background only', 'silhouette': 'silhouette'}
RAMP = ['#cde2fb', '#b7d3f6', '#9ec5f4', '#86b6ef', '#6da7ec', '#5598e7', '#3987e5', '#2a78d6', '#256abf',
        '#1c5cab', '#184f95', '#104281', '#0d366b']   # one hue, light to dark (sequential)
SEQ = LinearSegmentedColormap.from_list('blue_seq', RAMP)
MM = 1 / 25.4
SINGLE, DOUBLE = 85 * MM, 174 * MM   # journal column widths

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6.5, 'axes.edgecolor': AXIS, 'axes.linewidth': 0.6, 'axes.labelcolor': INK,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 0, 'text.color': INK, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'legend.frameon': False, 'savefig.facecolor': 'white', 'figure.facecolor': 'white',
    'pdf.fonttype': 42, 'svg.fonttype': 'none'})

In [ ]:
# 4. Load and verify every input (each file must match its run record)
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

man = {nb: verify(IN[nb]) for nb in ('01', '02', '03', '04')}
for nb in ('05', '06', '07', '08', '09'):
    man[nb] = verify(IN[nb] / 'real')
    assert man[nb]['in_colab'] and not man[nb]['local_test_settings'], f'notebook {nb} real is not the Colab run'
    assert man[nb]['labels'] == 'real', f'notebook {nb}: expected the real-label run'
man['07 rehearsal'] = verify(IN['07'] / 'rehearsal')
man['07 decoding'] = verify(IN['07'] / 'decoding')
s07 = man['07']['settings']
assert (s07['SEED'], s07['N_TASK_PERM']) == (SEED, N_TASK_PERM), 'task-permutation settings differ from notebook 07'
INPUT_RUNS = {nb: m['run_started_utc'] for nb, m in man.items()}
print('Inputs verified:', ', '.join(INPUT_RUNS))

audit = pd.read_csv(IN['03'] / 'audit_primary.csv')
tasks_final = pd.read_csv(IN['02'] / 'tasks_final.csv')
groups_fg = pd.read_csv(IN['03'] / 'audit_feature_groups.csv')
sites = pd.read_csv(IN['03'] / 'audit_sites.csv')
labels = pd.read_csv(IN['02'] / 'labels_long.csv')
probe05 = pd.read_csv(IN['05'] / 'real' / 'image_probe.csv')
sil06 = pd.read_csv(IN['06'] / 'real' / 'silhouette_probe.csv')
sv = {m: pd.read_csv(IN['07'] / m / 'screen_validation_tasks.csv') for m in ('rehearsal', 'real')}
svs = {m: pd.read_csv(IN['07'] / m / 'screen_validation_summary.csv') for m in ('rehearsal', 'real')}
decoding = pd.read_csv(IN['07'] / 'decoding' / 'acquisition_decoding.csv')
transfer = pd.read_csv(IN['08'] / 'real' / 'transfer.csv')
meta_transfer = pd.read_csv(IN['08'] / 'real' / 'metadata_transfer.csv')
sil09 = pd.read_csv(IN['09'] / 'real' / 'silhouette_transfer.csv')
hb09 = pd.read_csv(IN['09'] / 'real' / 'header_body_transfer.csv')
fields09 = pd.read_csv(IN['09'] / 'real' / 'field_differences_numeric.csv')
print(f'{len(audit)} audited tasks, {int(audit.flagged.sum())} flagged; {sv["real"].task_id.nunique()} CT tasks in the '
      f'screen validation; {len(transfer)} transfer readings')

## Plan 7i: robustness of the screen validation (post hoc sensitivity analysis)
First the notebook 07 correlation over all 19 tasks is recomputed and must equal notebook 07's exactly (same function,
seed and permutations). Then the two checks and the leave-one-out range, for both encoders. The same is done on
notebook 07's rehearsal (shuffled image labels), where nothing should show.

In [ ]:
# 5. Plan 7i
def spearman_perm(a, b):
    """Notebook 07's function, unchanged: Spearman rho of two task-level vectors, with a one-sided test by random
    permutations of the tasks."""
    ok = ~(np.isnan(a) | np.isnan(b))
    ra, rb = rankdata(np.asarray(a)[ok]), rankdata(np.asarray(b)[ok])
    ra = ra - ra.mean()
    obs = float(ra @ (rb - rb.mean()) / np.sqrt((ra ** 2).sum() * ((rb - rb.mean()) ** 2).sum()))
    rng = np.random.default_rng(SEED)
    null = np.empty(N_TASK_PERM)
    for i0 in range(0, N_TASK_PERM, 10_000):
        P = np.array([rng.permutation(rb) for _ in range(min(10_000, N_TASK_PERM - i0))])
        P = P - P.mean(1, keepdims=True)
        null[i0:i0 + len(P)] = (P @ ra) / np.sqrt((P ** 2).sum(1) * (ra ** 2).sum())
    return obs, (1 + int((null >= obs - 1e-12).sum())) / (1 + N_TASK_PERM), int(ok.sum())

def rho_only(a, b):
    ra, rb = rankdata(a), rankdata(b)
    ra, rb = ra - ra.mean(), rb - rb.mean()
    return float(ra @ rb / np.sqrt((ra ** 2).sum() * (rb ** 2).sum()))

# Groups of tasks that share no patients (connected components of "shares at least one patient"; patient lists only)
TASKS19 = sorted(sv['real'].task_id.unique())
assert len(TASKS19) == 19 and sorted(sv['rehearsal'].task_id.unique()) == TASKS19
pts = {t: set(zip(g.collection, g.PatientID)) for t, g in labels[labels.task_id.isin(TASKS19)].groupby('task_id')}
assert sorted(pts) == TASKS19
parent = {t: t for t in TASKS19}
def root(t):
    while parent[t] != t:
        t = parent[t]
    return t
for i, a in enumerate(TASKS19):
    for b in TASKS19[i + 1:]:
        if pts[a] & pts[b]:
            parent[root(a)] = root(b)
comp = {}
for t in TASKS19:
    comp.setdefault(root(t), []).append(t)
task_groups = pd.DataFrame([{'task_id': t, 'group': min(m), 'group_size': len(m)} for m in comp.values() for t in m])
task_groups = task_groups.sort_values(['group', 'task_id']).reset_index(drop=True)
print(f'{len(comp)} groups of tasks that share no patients')
show(task_groups)

def wording(check, rho, p):
    ok = rho > 0 and p < 0.05
    if check == 'without the flagged tasks':
        return 'the correlation holds without the flagged tasks' if ok else 'the correlation depends on the flagged tasks'
    return 'holds over patient-disjoint task groups' if ok else 'not shown over patient-disjoint task groups'

rob, loo = [], []
for mode in ('rehearsal', 'real'):
    for enc in ENCODERS:
        d = sv[mode][sv[mode].encoder == enc].set_index('task_id')   # notebook 07's row order (the test depends on it)
        assert sorted(d.index) == TASKS19
        rec = svs[mode].query('encoder == @enc and input == "background"').iloc[0]
        rho, p, n = spearman_perm(d.metadata_auc.to_numpy(), d.background_auc.to_numpy())
        same = abs(rho - rec.spearman_rho) < 1e-12 and abs(p - rec.perm_p_one_sided) < 1e-12
        rob.append({'mode': mode, 'encoder': enc, 'check': 'all 19 tasks (notebook 07, recomputed)', 'n_points': n,
                    'spearman_rho': rho, 'perm_p_one_sided': p, 'wording': rec.reading, 'reproduces_nb07': same})
        u = d[~d.metadata_flagged.astype(bool)]
        rho, p, n = spearman_perm(u.metadata_auc.to_numpy(), u.background_auc.to_numpy())
        rob.append({'mode': mode, 'encoder': enc, 'check': 'without the flagged tasks', 'n_points': n,
                    'spearman_rho': rho, 'perm_p_one_sided': p, 'wording': wording('without the flagged tasks', rho, p)})
        gm = d.join(task_groups.set_index('task_id')).groupby('group')[['metadata_auc', 'background_auc']].mean()
        rho, p, n = spearman_perm(gm.metadata_auc.to_numpy(), gm.background_auc.to_numpy())
        rob.append({'mode': mode, 'encoder': enc, 'check': 'patient-disjoint task groups (group means)', 'n_points': n,
                    'spearman_rho': rho, 'perm_p_one_sided': p, 'wording': wording('groups', rho, p)})
        full = rho_only(d.metadata_auc.to_numpy(), d.background_auc.to_numpy())
        for t in d.index:
            dd = d.drop(index=t)
            r = rho_only(dd.metadata_auc.to_numpy(), dd.background_auc.to_numpy())
            loo.append({'mode': mode, 'encoder': enc, 'task_left_out': t, 'metadata_flagged': bool(d.loc[t, 'metadata_flagged']),
                        'spearman_rho': r, 'change_from_all_19': r - full})
robust = pd.DataFrame(rob)
loo = pd.DataFrame(loo)
assert robust.reproduces_nb07.dropna().astype(bool).all(), 'notebook 07 correlation NOT reproduced: inspect the run log'
loo_summary = (loo.groupby(['mode', 'encoder'])
                  .apply(lambda g: pd.Series({'rho_min': g.spearman_rho.min(), 'rho_max': g.spearman_rho.max(),
                                              'largest_change_task': g.loc[g.change_from_all_19.abs().idxmax(), 'task_left_out'],
                                              'largest_change': g.loc[g.change_from_all_19.abs().idxmax(), 'change_from_all_19']}),
                         include_groups=False).reset_index())
task_groups.to_csv(OUT / 'task_groups.csv', index=False)
robust.to_csv(OUT / 'robustness_screen_validation.csv', index=False)
loo.to_csv(OUT / 'robustness_leave_one_out.csv', index=False)
loo_summary.to_csv(OUT / 'robustness_leave_one_out_summary.csv', index=False)
show(robust)
show(loo_summary)

## Figures
Every figure is saved as PNG (300 dpi) and PDF in `figures/`, and every number it shows is in one of the input
tables or in `tables/`.

In [ ]:
# 6. Figure helpers and short task names
SHORT = {'renal subtype (KIRC vs KIRP)': 'TCGA renal subtype (KIRC vs KIRP)',
         'lung histology (LUAD vs LUSC)': 'TCGA lung histology (LUAD vs LUSC)'}
def short(t):
    return SHORT.get(t, t)

def axis_style(ax, grid='x'):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    if grid:
        ax.grid(axis=grid, color=GRID, lw=0.5)
    ax.set_axisbelow(True)

def chance(ax, orient='v', flag_line=True):
    line = ax.axvline if orient == 'v' else ax.axhline
    line(0.5, color=MUTED, lw=0.7, ls=(0, (3, 2)), zorder=1)
    if flag_line:
        line(0.6, color=MUTED, lw=0.6, ls=(0, (1, 2)), zorder=1)

def ci_point(ax, x, lo, hi, y, color, filled=True, marker='o', size=18, lw=1.2):
    ax.plot([lo, hi], [y, y], color=color, lw=lw, solid_capstyle='round', zorder=2)
    ax.scatter([x], [y], s=size, marker=marker, facecolor=color if filled else 'white', edgecolor=color,
               linewidth=1.0 if not filled else 0.6, zorder=3)

SAVED = []
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = FIG / f'{name}.{ext}'
        fig.savefig(p, dpi=300, bbox_inches='tight', metadata={'CreationDate': None} if ext == 'pdf' else None)
        SAVED.append(p)
    plt.close(fig)
    print('saved', name)

def fmt(a, lo, hi):
    return f'{a:.3f} ({lo:.3f}–{hi:.3f})'

In [ ]:
# 7. Fig. 1: workflow
n_tasks, n_flag = len(audit), int(audit.flagged.sum())
n_ct = len(TASKS19)
steps = [('TCIA public API', 'series metadata only\nno image download'),
         ('1  Metadata screen', f'{n_tasks} benchmark tasks\n{n_flag} flagged\n(RQ1–RQ3)'),
         ('2  Patient-free test', f'background only and\nsilhouette, {n_ct} CT\ntasks (RQ4, RQ6)'),
         ('3  Consequence', 'external transfer:\nclean vs confounded\ntarget (RQ7)'),
         ('4  Mechanism', 'embeddings decode\nscanner settings\n(RQ8)')]
fig, ax = plt.subplots(figsize=(DOUBLE, 1.15))
ax.set_xlim(0, len(steps)); ax.set_ylim(0, 1); ax.axis('off')
for i, (head, body) in enumerate(steps):
    ax.add_patch(FancyBboxPatch((i + 0.06, 0.08), 0.84, 0.84, boxstyle='round,pad=0,rounding_size=0.06',
                                facecolor='#f4f8fe' if i else '#f3f2ee', edgecolor=AXIS, lw=0.6))
    ax.text(i + 0.48, 0.74, head, ha='center', va='center', fontsize=6.8, fontweight='bold', color=INK)
    ax.text(i + 0.48, 0.38, body, ha='center', va='center', fontsize=6.0, color=INK2, linespacing=1.3)
    if i < len(steps) - 1:
        ax.annotate('', xy=(i + 1.06, 0.5), xytext=(i + 0.9, 0.5),
                    arrowprops=dict(arrowstyle='-|>', color=MUTED, lw=0.8, mutation_scale=7))
save(fig, 'fig1_workflow')

In [ ]:
# 8. Fig. 2 (risk map) and Fig. 3 (forest plot): the metadata screen over all audited tasks
a = audit.merge(tasks_final[['task_id', 'modality']], on='task_id', how='left').sort_values('auc').reset_index(drop=True)
assert a.modality.notna().all()
fig, ax = plt.subplots(figsize=(DOUBLE * 0.78, 4.6))
for i, r in a.iterrows():
    c = ORANGE if r.flagged else BLUE
    ci_point(ax, r.auc, r.ci_low, r.ci_high, i, c, filled=True, marker='o' if r.modality == 'CT' else 's', size=16)
chance(ax)
ax.set_yticks(range(len(a)), [short(t) for t in a.task_id])
for lab, r in zip(ax.get_yticklabels(), a.itertuples()):
    lab.set_color(INK if r.flagged else INK2); lab.set_fontweight('bold' if r.flagged else 'normal')
ax.set_ylim(-0.7, len(a) - 0.3)
ax.set_xlim(0.2, 1.0)
ax.set_xlabel('Label predicted from acquisition metadata only: AUC (95% CI)')
axis_style(ax)
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=ORANGE, label='flagged (lower CI ≥ 0.60 and q < 0.05)', ms=4),
                   Line2D([], [], marker='o', ls='', color=BLUE, label='not flagged', ms=4),
                   Line2D([], [], marker='o', ls='', mfc='white', mec=INK2, label='CT task', ms=4),
                   Line2D([], [], marker='s', ls='', mfc='white', mec=INK2, label='MR task', ms=4)],
          loc='lower right', handletextpad=0.3, borderaxespad=0.2)
save(fig, 'fig3_metadata_screen_forest')

GROUP_ORDER = ['scanner (vendor, model)', 'software version', 'image count', 'slice thickness and kernel', 'body part',
               'description words']
GROUP_LABEL = ['Scanner\n(vendor, model)', 'Software\nversion', 'Image\ncount', 'Slice thickness\nand kernel',
               'Body part', 'Description\nwords']
assert sorted(groups_fg.group.unique()) == sorted(GROUP_ORDER)
M = groups_fg.pivot(index='task_id', columns='group', values='alone_auc').loc[a.task_id, GROUP_ORDER]
fig, ax = plt.subplots(figsize=(DOUBLE * 0.82, 4.6))
vals = M.to_numpy(dtype=float)
im = ax.imshow(np.clip(vals, 0.5, 0.9), cmap=SEQ, vmin=0.5, vmax=0.9, aspect='auto', origin='lower')
for (i, j), v in np.ndenumerate(vals):
    if np.isnan(v):
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor='white', edgecolor=GRID, hatch='////', lw=0))
    else:
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5.6,
                color='white' if v >= 0.72 else INK)
ax.set_xticks(range(len(GROUP_ORDER)), GROUP_LABEL, fontsize=6.2)
ax.xaxis.tick_top()
ax.set_yticks(range(len(a)), [short(t) for t in a.task_id])
for lab, r in zip(ax.get_yticklabels(), a.itertuples()):
    lab.set_color(INK if r.flagged else INK2); lab.set_fontweight('bold' if r.flagged else 'normal')
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(np.arange(-0.5, len(GROUP_ORDER)), minor=True)
ax.set_yticks(np.arange(-0.5, len(a)), minor=True)
ax.grid(which='minor', color='white', lw=1.2)
ax.tick_params(which='minor', length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, ticks=[0.5, 0.6, 0.7, 0.8, 0.9])
cb.set_label('AUC of the field group alone (≤ 0.5 shown as 0.5)', fontsize=6.3)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6, length=2)
save(fig, 'fig2_risk_map_field_groups')

In [ ]:
# 9. Fig. 4: site or protocol? Overall, site only, whole sites held out, and within site
s = sites.sort_values('overall_auc_same_patients').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(SINGLE * 1.45, 3.4))
offs = {'overall': 0.27, 'site only': 0.09, 'sites held out': -0.09, 'within site': -0.27}
col = {'overall': INK, 'site only': BLUE, 'sites held out': ORANGE, 'within site': AQUA}
for i, r in s.iterrows():
    ax.scatter([r.overall_auc_same_patients], [i + offs['overall']], s=14, marker='D', color=INK, zorder=3)
    ci_point(ax, r.site_only_auc, r.site_only_ci_low, r.site_only_ci_high, i + offs['site only'], BLUE, size=12, lw=1)
    ci_point(ax, r.site_heldout_auc, r.site_heldout_ci_low, r.site_heldout_ci_high, i + offs['sites held out'], ORANGE,
             size=12, lw=1)
    ci_point(ax, r.within_site_auc, r.within_site_ci_low, r.within_site_ci_high, i + offs['within site'], AQUA, size=12,
             lw=1)
    if i:
        ax.axhline(i - 0.5, color=GRID, lw=0.5)
chance(ax, flag_line=False)
ax.set_yticks(range(len(s)), [short(t) for t in s.task_id])
ax.set_ylim(-0.6, len(s) - 0.4)
ax.set_xlim(0, 1)
ax.set_xlabel('AUC (95% CI)')
axis_style(ax, grid=None)
ax.legend(handles=[Line2D([], [], marker='D', ls='', color=INK, ms=3.5, label='metadata, all patients'),
                   Line2D([], [], marker='o', ls='', color=BLUE, ms=3.5, label='submitting site alone'),
                   Line2D([], [], marker='o', ls='', color=ORANGE, ms=3.5, label='metadata, whole sites held out'),
                   Line2D([], [], marker='o', ls='', color=AQUA, ms=3.5, label='metadata, pairs within one site')],
          loc='upper center', bbox_to_anchor=(0.45, -0.13), ncol=2, handletextpad=0.2, columnspacing=1.0)
save(fig, 'fig4_site_vs_protocol')

In [ ]:
# 10. Fig. 5: diagnosis without the patient, the four flagged tasks
p5 = probe05.merge(sil06[['task_id', 'encoder', 'silhouette_auc', 'silhouette_ci_low', 'silhouette_ci_high',
                          'silhouette_flagged']], on=['task_id', 'encoder'], how='left', validate='one_to_one')
assert p5.silhouette_auc.notna().all()
order5 = a[a.flagged].sort_values('auc').task_id.tolist()
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.55), sharey=True)
for ax, enc in zip(axes, ENCODERS):
    d = p5[p5.encoder == enc].set_index('task_id').loc[order5]
    for i, (t, r) in enumerate(d.iterrows()):
        ax.scatter([r.metadata_auc_same_patients], [i + 0.3], s=13, marker='D', color=INK, zorder=3)
        ci_point(ax, r.image_auc, r.image_ci_low, r.image_ci_high, i + 0.1, BLUE, filled=bool(r.image_flagged), size=15)
        ci_point(ax, r.background_auc, r.background_ci_low, r.background_ci_high, i - 0.1, ORANGE,
                 filled=bool(r.background_flagged), size=15)
        ci_point(ax, r.silhouette_auc, r.silhouette_ci_low, r.silhouette_ci_high, i - 0.3, AQUA,
                 filled=bool(r.silhouette_flagged), size=15)
        if i:
            ax.axhline(i - 0.5, color=GRID, lw=0.5)
    chance(ax)
    ax.set_title(f'{"ab"[ENCODERS.index(enc)]}  {ENC[enc]}')
    ax.set_xlim(0.15, 1.0)
    ax.set_ylim(-0.55, len(order5) - 0.45)
    ax.set_xlabel('AUC (95% CI)')
    axis_style(ax, grid=None)
axes[0].set_yticks(range(len(order5)), [short(t) for t in order5])
fig.legend(handles=[Line2D([], [], marker='D', ls='', color=INK, ms=3.5, label='metadata only'),
                    Line2D([], [], marker='o', ls='', color=BLUE, ms=4, label='full image'),
                    Line2D([], [], marker='o', ls='', color=ORANGE, ms=4, label='background only (patient removed)'),
                    Line2D([], [], marker='o', ls='', color=AQUA, ms=4, label='silhouette (body outline only)'),
                    Line2D([], [], marker='o', ls='', mfc='white', mec=INK2, ms=4, label='open: not flagged')],
           loc='lower center', bbox_to_anchor=(0.55, -0.13), ncol=5, handletextpad=0.2, columnspacing=0.9)
fig.subplots_adjust(wspace=0.06)
save(fig, 'fig5_diagnosis_without_patient')

In [ ]:
# 11. Fig. 6: does the screen find where patient-free images predict the label? (RQ6, with plan 7i)
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 3.7))
LABEL_POS = {('ctfm', 'LIDC-IDRI | primary vs metastatic'): (-6, 5, 'bottom'),
             ('ctfm', 'lung histology (LUAD vs LUSC)'): (-6, -6, 'top'),
             ('ctfm', 'CPTAC-PDA | stage'): (-6, -5, 'top'), ('merlin', 'CPTAC-PDA | stage'): (-6, -5, 'top')}
LABEL6 = {'renal subtype (KIRC vs KIRP)': 'KIRC vs KIRP', 'lung histology (LUAD vs LUSC)': 'LUAD vs LUSC',
          'LIDC-IDRI | primary vs metastatic': 'LIDC primary vs metastatic', 'CPTAC-PDA | stage': 'CPTAC-PDA stage'}
for ax, enc in zip(axes, ENCODERS):
    d = sv['real'][sv['real'].encoder == enc]
    for flag in (False, True):
        q = d[d.metadata_flagged.astype(bool) == flag]
        ax.scatter(q.metadata_auc, q.background_auc, s=24, color=ORANGE if flag else BLUE, edgecolor='white',
                   linewidth=0.6, zorder=3)
    bf = d[d.background_flagged.astype(bool)]
    ax.scatter(bf.metadata_auc, bf.background_auc, s=62, facecolor='none', edgecolor=INK, linewidth=0.8, zorder=4)
    for r in d[d.metadata_flagged.astype(bool)].itertuples():
        dx, dy, va = LABEL_POS.get((enc, r.task_id), (-7, 0, 'center'))
        ax.annotate(LABEL6.get(r.task_id, r.task_id), (r.metadata_auc, r.background_auc), xytext=(dx, dy),
                    textcoords='offset points', ha='right', va=va, fontsize=6, color=INK2)
    ax.axvline(0.5, color=MUTED, lw=0.7, ls=(0, (3, 2)), zorder=1)
    ax.axhline(0.5, color=MUTED, lw=0.7, ls=(0, (3, 2)), zorder=1)
    ax.set_xlim(0.3, 0.9); ax.set_ylim(0.3, 0.9); ax.set_aspect('equal')
    ax.set_xlabel('Metadata-only AUC (notebook 03)')
    if enc == 'ctfm':
        ax.set_ylabel('Background-only image AUC')
    rr = robust[(robust['mode'] == 'real') & (robust.encoder == enc)].set_index('check')
    lines = [('all 19 tasks', 'all 19 tasks (notebook 07, recomputed)'),
             ('15 unflagged tasks', 'without the flagged tasks'),
             ('11 patient-disjoint groups', 'patient-disjoint task groups (group means)')]
    txt = '\n'.join(f'{lab}: ρ = {rr.loc[k, "spearman_rho"]:.2f}, p = {rr.loc[k, "perm_p_one_sided"]:.3f}'
                    for lab, k in lines)
    ax.text(0.0, -0.2, 'Spearman ρ, one-sided permutation p\n' + txt, transform=ax.transAxes, ha='left', va='top',
            fontsize=6, color=INK2, linespacing=1.45)
    ax.set_title(f'{"ab"[ENCODERS.index(enc)]}  {ENC[enc]}{" (main encoder)" if enc == "ctfm" else ""}')
    axis_style(ax, grid='both')
fig.legend(handles=[Line2D([], [], marker='o', ls='', color=ORANGE, ms=4.5, label='flagged by the metadata screen'),
                    Line2D([], [], marker='o', ls='', color=BLUE, ms=4.5, label='not flagged'),
                    Line2D([], [], marker='o', ls='', mfc='none', mec=INK, ms=7, label='background-only probe flagged')],
           loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=3, handletextpad=0.2)
fig.subplots_adjust(wspace=0.22)
save(fig, 'fig6_screen_validation')

In [ ]:
# 12. Fig. 7: internal against external AUC (experiment B), and the CPTAC post hoc checks (plan 7h)
PAIRS = [('lung: TCGA -> NSCLC Radiogenomics', 'a  TCGA lung → NSCLC Radiogenomics (independent)'),
         ('lung: TCGA -> CPTAC', 'b  TCGA lung → CPTAC lung (built the same way)'),
         ('kidney: TCGA -> C4KC-KiTS', 'c  TCGA kidney → C4KC-KiTS (independent)')]
assert sorted(transfer.pair.unique()) == sorted(p for p, _ in PAIRS)
ROWS7 = [('ctfm', 'image'), ('ctfm', 'background'), ('merlin', 'image'), ('merlin', 'background')]
fig, axes = plt.subplots(2, 2, figsize=(DOUBLE, 4.1))
for ax, (pair, title) in zip(axes.flat[:3], PAIRS):
    d = transfer[transfer.pair == pair].set_index(['encoder', 'input'])
    for i, key in enumerate(ROWS7[::-1]):
        r = d.loc[key]
        ci_point(ax, r.source_internal_auc, r.source_internal_ci_low, r.source_internal_ci_high, i + 0.22, MUTED,
                 marker='D', size=12, lw=1)
        ci_point(ax, r.external_auc, r.external_ci_low, r.external_ci_high, i, INPUT_COLOR[key[1]],
                 filled=r.reading != 'does not transfer', size=17)
        ci_point(ax, r.target_internal_auc, r.target_internal_ci_low, r.target_internal_ci_high, i - 0.22, INK,
                 marker='s', filled=False, size=12, lw=1)
        if i:
            ax.axhline(i - 0.5, color=GRID, lw=0.5)
    chance(ax, flag_line=False)
    ax.set_yticks(range(4), [f'{ENC[e]}, {INPUT_NAME[k]}' for e, k in ROWS7[::-1]])
    ax.set_xlim(0.2, 1.0); ax.set_ylim(-0.55, 3.55)
    ax.set_title(title)
    axis_style(ax, grid=None)
ax = axes.flat[3]
rows_d = []
for r in sil09[sil09.target == 'CPTAC lung'].itertuples():
    e = transfer[(transfer.pair == 'lung: TCGA -> CPTAC') & (transfer.encoder == r.encoder) & (transfer.input == 'background')].iloc[0]
    rows_d.append((f'{ENC[r.encoder]}, background only', e.external_auc, e.external_ci_low, e.external_ci_high, ORANGE,
                   e.reading != 'does not transfer'))
for r in sil09[sil09.target == 'CPTAC lung'].itertuples():
    rows_d.append((f'{ENC[r.encoder]}, silhouette', r.silhouette_external_auc, r.silhouette_ci_low, r.silhouette_ci_high,
                   AQUA, bool(r.silhouette_transfers)))
h = hb09[hb09.target == 'CPTAC lung'].iloc[0]
rows_d.append(('DICOM header fields', h.header_external_auc, h.header_ci_low, h.header_ci_high, INK2, bool(h.header_transfers)))
rows_d.append(('body size', h.body_size_external_auc, h.body_size_ci_low, h.body_size_ci_high, INK2,
               bool(h.body_size_transfers)))
for i, (lab, v, lo, hi, c, filled) in enumerate(rows_d[::-1]):
    ci_point(ax, v, lo, hi, i, c, filled=filled, size=17)
chance(ax, flag_line=False)
ax.set_yticks(range(len(rows_d)), [r[0] for r in rows_d[::-1]])
ax.set_xlim(0.2, 1.0); ax.set_ylim(-0.6, len(rows_d) - 0.4)
ax.set_title('d  TCGA lung → CPTAC lung: what transfers? (post hoc)')
axis_style(ax, grid=None)
for ax in axes[1]:
    ax.set_xlabel('AUC (95% CI)')
fig.legend(handles=[Line2D([], [], marker='D', ls='', color=MUTED, ms=3.5, label='source, internal (CV)'),
                    Line2D([], [], marker='o', ls='', color=BLUE, ms=4, label='external: full image'),
                    Line2D([], [], marker='o', ls='', color=ORANGE, ms=4, label='external: background only'),
                    Line2D([], [], marker='o', ls='', color=AQUA, ms=4, label='external: silhouette'),
                    Line2D([], [], marker='s', ls='', mfc='white', mec=INK, ms=3.5, label='target, internal (CV)'),
                    Line2D([], [], marker='o', ls='', mfc='white', mec=INK2, ms=4, label='open: does not transfer')],
           loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=6, handletextpad=0.2, columnspacing=0.8)
fig.subplots_adjust(hspace=0.42, wspace=0.62)
save(fig, 'fig7_external_transfer')

In [ ]:
# 13. Supplementary figures: acquisition decoding (experiment C) and fields by class in the three lung datasets (7h)
TARGETS_C = decoding.target.unique().tolist()
cols = [(e, k) for e in ENCODERS for k in ('image', 'background', 'silhouette')]
D = decoding.set_index(['target', 'encoder', 'input'])
vals = np.array([[D.loc[(t, e, k), 'pooled_auc'] for e, k in cols] for t in TARGETS_C], dtype=float)
fig, ax = plt.subplots(figsize=(SINGLE * 1.5, 0.32 * len(TARGETS_C) + 0.9))
im = ax.imshow(np.clip(vals, 0.5, 1.0), cmap=SEQ, vmin=0.5, vmax=1.0, aspect='auto')
for (i, j), v in np.ndenumerate(vals):
    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color='white' if v >= 0.8 else INK)
ax.set_xticks(range(len(cols)), [f'{ENC[e]}\n{INPUT_NAME[k]}' for e, k in cols], fontsize=6)
ax.set_yticks(range(len(TARGETS_C)), TARGETS_C)
ax.xaxis.tick_top(); ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xticks(np.arange(-0.5, len(cols)), minor=True); ax.set_yticks(np.arange(-0.5, len(TARGETS_C)), minor=True)
ax.grid(which='minor', color='white', lw=1.2); ax.tick_params(which='minor', length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, ticks=[0.5, 0.75, 1.0])
cb.set_label('Decoding AUC, pooled over collections', fontsize=6.3)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6, length=2)
save(fig, 'figS1_acquisition_decoding')

DS = [('lung histology (LUAD vs LUSC)', 'TCGA lung', BLUE), ('CPTAC lung', 'CPTAC lung', ORANGE),
      ('NSCLC Radiogenomics', 'NSCLC Radiogenomics', AQUA)]
FIELD_NAME = {'hdr_slice_thickness': 'slice thickness', 'hdr_z_spacing': 'slice spacing',
              'hdr_recon_diameter': 'reconstruction diameter', 'hdr_kvp': 'kVp', 'hdr_tube_current': 'tube current',
              'body_area_cm2': 'body cross-section area'}
fig, ax = plt.subplots(figsize=(SINGLE * 1.3, 2.6))
fl = list(FIELD_NAME)[::-1]
for i, f in enumerate(fl):
    for k, (ds, lab, c) in enumerate(DS):
        r = fields09[(fields09.dataset == ds) & (fields09.field == f)].iloc[0]
        ci_point(ax, r.auc_squamous_higher, r.ci_low, r.ci_high, i + 0.22 - 0.22 * k, c, size=13, lw=1)
    if i:
        ax.axhline(i - 0.5, color=GRID, lw=0.5)
chance(ax, flag_line=False)
ax.set_yticks(range(len(fl)), [FIELD_NAME[f] for f in fl])
ax.set_ylim(-0.55, len(fl) - 0.45); ax.set_xlim(0.15, 0.9)
ax.set_xlabel('AUC for squamous having the higher value (95% CI)')
axis_style(ax, grid=None)
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=c, ms=4, label=lab) for _, lab, c in DS],
          loc='upper center', bbox_to_anchor=(0.4, -0.2), ncol=3, handletextpad=0.2)
save(fig, 'figS2_lung_fields_by_class')

## Tables
Formatted for the manuscript (AUC with 95% CI to three decimals), one CSV each in `tables/`.

In [ ]:
# 14. Tables
t1 = a.sort_values('auc', ascending=False).merge(tasks_final[['task_id', 'collections', 'label_source']], on='task_id',
                                                  how='left')
table1 = pd.DataFrame({
    'Task': t1.task_id, 'Collection(s)': t1.collections, 'Modality': t1.modality, 'Label': t1.label_source,
    'Type': t1.task_type, 'Patients': t1.n_patients, 'Classes': t1.classes,
    'Metadata-only AUC (95% CI)': [fmt(r.auc, r.ci_low, r.ci_high) for r in t1.itertuples()],
    'Permutation q (BH)': t1.perm_q_bh.round(3), 'Flagged': np.where(t1.flagged, 'yes', 'no')})

t2 = p5.set_index('task_id').loc[order5[::-1]].reset_index()     # highest metadata AUC first, both encoders
table2 = pd.DataFrame({
    'Task': t2.task_id, 'Encoder': t2.encoder.map(ENC), 'Patients': t2.n_patients,
    'Metadata only, same patients': t2.metadata_auc_same_patients.round(3),
    'Full image': [fmt(r.image_auc, r.image_ci_low, r.image_ci_high) + (' *' if r.image_flagged else '') for r in t2.itertuples()],
    'Background only': [fmt(r.background_auc, r.background_ci_low, r.background_ci_high) + (' *' if r.background_flagged else '')
                        for r in t2.itertuples()],
    'Silhouette': [fmt(r.silhouette_auc, r.silhouette_ci_low, r.silhouette_ci_high) + (' *' if r.silhouette_flagged else '')
                   for r in t2.itertuples()],
    'Body size only': [fmt(r.body_size_auc, r.body_size_ci_low, r.body_size_ci_high) for r in t2.itertuples()]})

s_real = svs['real'].copy()
rr = robust[robust['mode'] == 'real']
table3 = pd.concat([
    pd.DataFrame({'Encoder': s_real.encoder.map(ENC), 'Input': s_real.input.map(INPUT_NAME), 'Role': s_real.role,
                  'Tasks': s_real.n_tasks, 'Spearman rho': s_real.spearman_rho.round(2),
                  'One-sided p': s_real.perm_p_one_sided.round(4),
                  'Flagged by both / metadata only / image only / neither':
                      [f'{r.both_flagged} / {r.metadata_only_flagged} / {r.image_test_only_flagged} / {r.neither}'
                       for r in s_real.itertuples()],
                  'Reading': s_real.reading.fillna('(descriptive)')}),
    pd.DataFrame({'Encoder': rr.encoder.map(ENC), 'Input': 'background only',
                  'Role': 'robustness (7i, post hoc): ' + rr.check, 'Tasks': rr.n_points,
                  'Spearman rho': rr.spearman_rho.round(2), 'One-sided p': rr.perm_p_one_sided.round(4),
                  'Flagged by both / metadata only / image only / neither': '',
                  'Reading': rr.wording})[rr.check != 'all 19 tasks (notebook 07, recomputed)']],
    ignore_index=True)

t4 = transfer.copy()
t4['order'] = t4.pair.map({p: i for i, (p, _) in enumerate(PAIRS)})
t4 = t4.sort_values(['order', 'encoder', 'input'], ascending=[True, True, False])
table4 = pd.DataFrame({
    'Pair': t4.pair, 'Encoder': t4.encoder.map(ENC), 'Input': t4.input.map(INPUT_NAME),
    'Source internal AUC': [fmt(r.source_internal_auc, r.source_internal_ci_low, r.source_internal_ci_high) for r in t4.itertuples()],
    'External AUC': [fmt(r.external_auc, r.external_ci_low, r.external_ci_high) for r in t4.itertuples()],
    'External permutation p': t4.external_perm_p.round(3),
    'Gap (source − external)': [f'{r.transfer_gap:.3f} ({r.gap_ci_low:.3f} to {r.gap_ci_high:.3f})' for r in t4.itertuples()],
    'Target internal AUC': [fmt(r.target_internal_auc, r.target_internal_ci_low, r.target_internal_ci_high) for r in t4.itertuples()],
    'Reading': t4.reading})

table5 = pd.DataFrame(
    [{'Model fitted on TCGA lung': lab, 'External AUC on CPTAC lung (95% CI)': fmt(v, lo, hi),
      'Transfers': 'yes' if f else 'no'} for lab, v, lo, hi, _, f in rows_d])
sil_c = sil09[sil09.target == 'CPTAC lung']
table5_note = '; '.join(f'{ENC[r.encoder]}: background − silhouette {r.background_minus_silhouette:.3f} '
                        f'({r.difference_ci_low:.3f} to {r.difference_ci_high:.3f}), reading "{r.reading}"'
                        for r in sil_c.itertuples()) + f'; header and body size: "{h.reading}"'

tables = {'table1_metadata_screen.csv': table1, 'table2_patient_free_probes.csv': table2,
          'table3_screen_validation.csv': table3, 'table4_external_transfer.csv': table4,
          'table5_cptac_posthoc.csv': table5}
for name, t in tables.items():
    t.to_csv(TAB / name, index=False, encoding='utf-8-sig')   # opens correctly in Excel (dashes, rho)
(TAB / 'table5_cptac_posthoc_note.txt').write_text(table5_note + '\n', encoding='utf-8')
for name, t in tables.items():
    print('\n' + name); show(t)
print('\n' + table5_note)

In [ ]:
# 15. Run record
outputs = sorted(q for q in OUT.rglob('*') if q.is_file() and q.name != 'run_manifest.json')
manifest = {
    'notebook': '10_figures_tables.ipynb', 'plan': '7i (post hoc sensitivity analysis) and section 8',
    'run_started_utc': RUN_STARTED,
    'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB, 'input_runs': INPUT_RUNS, 'versions': VERSIONS,
    'settings': {'SEED': SEED, 'N_TASK_PERM': N_TASK_PERM},
    'reproduced_notebook07_correlation': bool(robust.reproduces_nb07.dropna().astype(bool).all()),
    'n_task_groups': len(comp),
    'outputs': [{'file': q.relative_to(OUT).as_posix(), 'bytes': q.stat().st_size, 'md5': md5(q)} for q in outputs]}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=_jsonable))
print(f'{len(outputs)} files written to {OUT}')

In [ ]:
# 16. Plain-English summary
print('PLAN 7i, real labels (post hoc sensitivity analysis; wording fixed in the plan before computing):')
for r in robust[robust['mode'] == 'real'].itertuples():
    print(f'- {ENC[r.encoder]}, {r.check}: rho {r.spearman_rho:.2f}, one-sided p {r.perm_p_one_sided:.4f} '
          f'({r.n_points} points) -> {r.wording}')
for r in loo_summary[loo_summary['mode'] == 'real'].itertuples():
    print(f'- {ENC[r.encoder]}, leave one task out: rho {r.rho_min:.2f} to {r.rho_max:.2f}; the largest change '
          f'({r.largest_change:+.2f}) comes from leaving out {r.largest_change_task}')
print('\nRehearsal (shuffled image labels in notebook 07; nothing should show):')
for r in robust[robust['mode'] == 'rehearsal'].itertuples():
    print(f'- {ENC[r.encoder]}, {r.check}: rho {r.spearman_rho:.2f}, p {r.perm_p_one_sided:.4f}')
print('\nFigures:', ', '.join(sorted({p.stem for p in SAVED})))
print('Tables :', ', '.join(tables))

In [ ]:
# 17. Stop the runtime once everything above has finished (Colab only)
if IN_COLAB:
    from google.colab import runtime
    print('Notebook finished:', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
    drive.flush_and_unmount()
    print('Drive synced. Stopping the runtime now.')
    time.sleep(5)
    runtime.unassign()